# Notebook 10: Temporal Reinforcement Learning (PPO) Integration

**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Phase:** Temporal Reinforcement Learning (PPO Integration)  
**Author:** Defensive Cybersecurity ML Research Team  

---

> [!CAUTION]
> **CRITICAL RESEARCH NOTICE & DEFENSIVE SAFETY BOUNDARY**:
> **Temporal baseline execution is a pipeline validation experiment using synthetic temporal data, not a real ransomware detection evaluation.**
> In this phase, we connect the temporal dataset pipeline with a Gymnasium-compatible reinforcement learning environment (`TemporalRansomwareEnv`) and train a Proximal Policy Optimization (PPO) agent.
> - Observations at window $t$ consist strictly of causal behavioral, memory, delta, and rolling features available up to window $t$.
> - The old synthetic "feature reveal" mechanism from Notebook 04/05 is replaced by sequential temporal window progression ($W_0 \to W_1 \to W_2 \to W_3 \to W_4$).
> - NO ransomware binaries are downloaded, executed, or created.

## Section 1: Inspect Pipeline Assets & Load Temporal Dataset

We load the engineered temporal dataset and the predictive features curated in Phase 10.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np

# Load predictive features
with open('data/processed/temporal_model_features.json', 'r') as f:
    feature_names = json.load(f)

# Load engineered temporal demonstration telemetry
import src.train_temporal_baseline as ttb
df_features, val_report = ttb.load_and_prepare_temporal_dataset()

print("=" * 70)
print("TEMPORAL TELEMETRY LOADED FOR PPO INTEGRATION")
print("=" * 70)
print(f"Total Rows: {len(df_features)} across {df_features['sample_id'].nunique()} samples")
print(f"Predictive Feature Dimension (State Space): {len(feature_names)}")
print(f"Observation Windows: {sorted(df_features['observation_window'].unique())}")
print(f"Samples Present: {df_features['sample_id'].unique().tolist()}")
print("=" * 70)

TEMPORAL TELEMETRY LOADED FOR PPO INTEGRATION
Total Rows: 40 across 8 samples
Predictive Feature Dimension (State Space): 60
Observation Windows: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Samples Present: ['SAMPLE_001', 'SAMPLE_002', 'SAMPLE_003', 'SAMPLE_004', 'SAMPLE_005', 'SAMPLE_006', 'SAMPLE_007', 'SAMPLE_008']


## Section 2: Temporal PPO Environment & Reward Structure

### Observation Space
At discrete window $t$, the agent receives:
$$\mathbf{s}_t = [\text{behavior features}, \Delta \text{features}, \text{rolling moments}, \text{memory features}, \Delta \text{memory}]$$
Dimension: $d = 60$. Normalized using frozen parameters fitted strictly on training samples.

### Action Space
- `0 = CONTINUE_MONITORING`: Pay nominal monitoring cost $\rho = -0.1$, advance to window $t+1$.
- `1 = SUSPICIOUS`: Flag process under suspicion (+0.5 if ransomware, -0.5 if benign), advance to window $t+1$.
- `2 = RANSOMWARE_ALERT`: Immediate early stopping action. Quarantines process and terminates episode.
  - Correct alert: $+10.0 + 2.5 \times (4 - t)$ (early detection bonus).
  - False alarm: $-10.0$ penalty.

In [2]:
from src.temporal_ppo_environment import (
    TemporalRansomwareEnv,
    ACTION_NAMES,
    DEFAULT_TEMPORAL_REWARD_CONFIG
)

print("=" * 65)
print("TEMPORAL REINFORCEMENT LEARNING ENVIRONMENT CONFIGURATION")
print("=" * 65)
print(f"Action Space (Discrete 3): {ACTION_NAMES}")
print("Reward Configuration:")
for k, v in DEFAULT_TEMPORAL_REWARD_CONFIG.items():
    print(f"  - {k:<36}: {v}")
print("=" * 65)

TEMPORAL REINFORCEMENT LEARNING ENVIRONMENT CONFIGURATION
Action Space (Discrete 3): {0: 'CONTINUE_MONITORING', 1: 'SUSPICIOUS', 2: 'RANSOMWARE_ALERT'}
Reward Configuration:
  - alert_true_positive                 : 10.0
  - early_detection_bonus_factor        : 2.5
  - alert_false_positive                : -10.0
  - monitoring_cost                     : -0.1
  - suspicious_ransomware_reward        : 0.5
  - suspicious_benign_penalty           : -0.5
  - final_missed_ransomware_penalty     : -15.0
  - final_benign_true_negative_reward   : 5.0


## Section 3: Train / Test Partitioning & Leakage Verification

All observation windows for a sample belong entirely to either the train or test set.

In [3]:
from src.temporal_pipeline import split_by_sample

# Split strictly by sample_id
train_df, test_df, split_summary = split_by_sample(
    df_features, train_ratio=0.75, seed=42, sample_col='sample_id', stratify_col='label'
)

train_sample_ids = split_summary['train_samples']
test_sample_ids = split_summary['test_samples']

# Programmatic Leakage Verification
overlap = set(train_sample_ids).intersection(set(test_sample_ids))
assert len(overlap) == 0, f"Sample overlap detected: {overlap}"

print("=" * 65)
print("PPO DATA SPLIT & LEAKAGE VERIFICATION")
print("=" * 65)
print(f"Train Sample IDs ({len(train_sample_ids)}): {train_sample_ids}")
print(f"Test Sample IDs  ({len(test_sample_ids)}):  {test_sample_ids}")
print(f"Sample Intersection: {overlap} (Strictly Empty Set)")
print("-" * 65)
print("PPO DATA LEAKAGE CHECK: PASS")
print("=" * 65)

PPO DATA SPLIT & LEAKAGE VERIFICATION
Train Sample IDs (6): ['SAMPLE_001', 'SAMPLE_002', 'SAMPLE_003', 'SAMPLE_004', 'SAMPLE_006', 'SAMPLE_007']
Test Sample IDs  (2):  ['SAMPLE_005', 'SAMPLE_008']
Sample Intersection: set() (Strictly Empty Set)
-----------------------------------------------------------------
PPO DATA LEAKAGE CHECK: PASS


## Section 4: Train PPO Agent with Stable-Baselines3

We train a PPO agent with an MLP policy strictly on the training partition.

In [4]:
import src.train_temporal_ppo as ttp

# Train PPO agent
ppo_model, train_env, episode_rewards = ttp.train_temporal_ppo(
    df=df_features,
    feature_names=feature_names,
    train_sample_ids=train_sample_ids,
    reward_config=DEFAULT_TEMPORAL_REWARD_CONFIG,
    total_timesteps=15000,
    seed=42,
    model_save_path=project_root / 'models' / 'temporal' / 'ppo_temporal_model.zip'
)

print(f"Training completed successfully. Total recorded episodes: {len(episode_rewards)}")
print(f"Mean reward over last 20 episodes: {np.mean(episode_rewards[-20:]):.2f}")

Training Temporal PPO on 6 training samples for 15000 timesteps...
Saved trained PPO model checkpoint to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\models\temporal\ppo_temporal_model.zip
Training completed successfully. Total recorded episodes: 6497
Mean reward over last 20 episodes: 13.80


## Section 5: Evaluate PPO on Untouched Test Samples

We evaluate the policy deterministically on test samples:
- `SAMPLE_005`: Ransomware (Minority Family L)
- `SAMPLE_008`: Benign (Explorer Extension)

In [5]:
# Evaluate policy on test samples
ppo_metrics, test_traces_df = ttp.evaluate_temporal_ppo_policy(
    model=ppo_model,
    df=df_features,
    feature_names=feature_names,
    test_sample_ids=test_sample_ids,
    reward_config=DEFAULT_TEMPORAL_REWARD_CONFIG,
    seed=42,
)

print("=" * 65)
print("PPO SEQUENTIAL DECISION EVALUATION ON TEST SET")
print("=" * 65)
print(f"Correct Alert Rate (True Positives): {ppo_metrics['correct_alert_rate'] * 100:.1f}%")
print(f"False Alert Rate (False Alarms):    {ppo_metrics['false_alert_rate'] * 100:.1f}%")
print(f"Miss Rate (Undetected):              {ppo_metrics['miss_rate'] * 100:.1f}%")
print(f"Average Decision Window:             {ppo_metrics['average_decision_window']:.2f}")
print(f"Average Episode Reward:              {ppo_metrics['average_episode_reward']:.2f}")
print("=" * 65)

print("\nDetailed Test Episode Outcomes:")
for ep in ppo_metrics['episodes_summary']:
    print(f"  Sample: {ep['sample_id']} | Family: {ep['family']} | True: {'Ransomware' if ep['is_ransomware'] else 'Benign'} | Alerted: {ep['final_action_alert']} | Decision Window: W{ep['decision_window']} | Reward: {ep['total_reward']:.2f}")

print("\nDecision Traces Log:")
print(test_traces_df[['sample_id', 'family', 'window', 'action_name', 'reward', 'outcome']].to_string(index=False))

PPO SEQUENTIAL DECISION EVALUATION ON TEST SET
Correct Alert Rate (True Positives): 100.0%
False Alert Rate (False Alarms):    0.0%
Miss Rate (Undetected):              0.0%
Average Decision Window:             3.50
Average Episode Reward:              8.95

Detailed Test Episode Outcomes:
  Sample: SAMPLE_005 | Family: L | True: Ransomware | Alerted: True | Decision Window: W3 | Reward: 13.40
  Sample: SAMPLE_008 | Family: benign | True: Benign | Alerted: False | Decision Window: W4 | Reward: 4.50

Decision Traces Log:
 sample_id family  window         action_name  reward             outcome
SAMPLE_005      L       0 CONTINUE_MONITORING    -0.1 CONTINUE_MONITORING
SAMPLE_005      L       1          SUSPICIOUS     0.5 CONTINUE_SUSPICIOUS
SAMPLE_005      L       2          SUSPICIOUS     0.5 CONTINUE_SUSPICIOUS
SAMPLE_005      L       3    RANSOMWARE_ALERT    12.5 ALERT_TRUE_POSITIVE
SAMPLE_008 benign       0 CONTINUE_MONITORING    -0.1 CONTINUE_MONITORING
SAMPLE_008 benign       1 CONT

## Section 6: Action Distribution & Visualizations

We generate the training reward curve, confusion matrix, and action distribution plot.

In [6]:
# Generate figures, CSV reports, and markdown summary
ttp.generate_ppo_visualizations_and_reports(
    episode_rewards=episode_rewards,
    metrics=ppo_metrics,
    traces_df=test_traces_df,
    reward_config=DEFAULT_TEMPORAL_REWARD_CONFIG,
)

print("Visualizations and reports generated successfully.")

Saved PPO Training Curve to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\figures\temporal_ppo_training_curve.png
Saved PPO Confusion Matrix to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\figures\temporal_ppo_confusion_matrix.png
Saved PPO Action Distribution plot to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\figures\temporal_ppo_action_distribution.png
Saved PPO Decision Traces to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\ppo_decision_traces.csv
Saved Temporal PPO Config to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\temporal_ppo_config.json
Saved Temporal PPO Analysis to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\temporal_ppo_analysis.md
Visualizations and reports generated successfully.


## Section 7: Final Conclusion Statement

In [7]:
print("""Temporal PPO integration completed.

The agent successfully learned sequential stopping policies on temporal observations.

Temporal baseline execution is a pipeline validation experiment using synthetic temporal data,
not a real ransomware detection evaluation.

No real malware execution was performed.""")

Temporal PPO integration completed.

The agent successfully learned sequential stopping policies on temporal observations.

Temporal baseline execution is a pipeline validation experiment using synthetic temporal data,
not a real ransomware detection evaluation.

No real malware execution was performed.
